In [3]:
# Install
!pip install roboflow ultralytics

# Download dataset
from roboflow import Roboflow
rf = Roboflow(api_key="YOUT-ROBOFLOW-API-KEY")
project = rf.workspace("gaurang-patil-gr0h9").project("helmet-detection-ntbfz-yozkl")
version = project.version(2)
dataset = version.download("yolov8")

# Train
import torch
from ultralytics import YOLO

print(torch.cuda.get_device_name(0))

model = YOLO("yolov8m.pt")  # medium since Kaggle has more VRAM

model.train(
    data="/kaggle/working/helmet-detection-2/data.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    patience=10,
    name="helmet_kaggle_v1",
    pretrained=True,
    optimizer="Adam",
)

loading Roboflow workspace...
loading Roboflow project...
Tesla T4
Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/helmet-detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_kaggle_v1-2, nbs

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a67fc47f530>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [5]:
import os
for root, dirs, files in os.walk("/kaggle/working/runs"):
    for file in files:
        if file == "best.pt":
            print(os.path.join(root, file))

/kaggle/working/runs/detect/helmet_kaggle_v1-2/weights/best.pt


In [6]:
from ultralytics import YOLO

# Load best trained model
model = YOLO("/kaggle/working/runs/detect/helmet_kaggle_v1-2/weights/best.pt")

# Run evaluation on test set
metrics = model.val(
    data="/kaggle/working/helmet-detection-2/data.yaml",
    split="test",   # evaluate on test set specifically
    imgsz=640,
    batch=32,
    device=0,
)

print(f"mAP50:        {metrics.box.map50:.4f}")
print(f"mAP50-95:     {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1700.0±433.0 MB/s, size: 71.3 KB)
val: Scanning /kaggle/working/helmet-detection-2/test/labels... 462 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 462/462 1.2Kit/s 0.4s<0.1s
val: New cache created: /kaggle/working/helmet-detection-2/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 35, len(boxes) = 3125. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.1it/s 13.3s0.9s
                   all        462       3125      0.918      0.884      0.943      0.763
             bicyclist         32 

In [7]:
import shutil
import os

# Zip the entire runs folder
shutil.make_archive(
    "/kaggle/working/helmet_kaggle_results",  # output zip name
    "zip",
    "/kaggle/working/runs"                     # folder to zip
)

print("Zipped! Download from Kaggle output panel →")
print("Files included:")
for root, dirs, files in os.walk("/kaggle/working/runs/detect/helmet_kaggle_v1"):
    for file in files:
        print(os.path.join(root, file))

Zipped! Download from Kaggle output panel →
Files included:
/kaggle/working/runs/detect/helmet_kaggle_v1/args.yaml


In [8]:
import os
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".pt"):
            size = os.path.getsize(os.path.join(root, file))
            print(f"{os.path.join(root, file)} — {size/1024/1024:.1f} MB")

/kaggle/working/yolov8m.pt — 49.7 MB
/kaggle/working/yolo26n.pt — 5.3 MB
/kaggle/working/runs/detect/helmet_kaggle_v1-2/weights/best.pt — 49.6 MB
/kaggle/working/runs/detect/helmet_kaggle_v1-2/weights/last.pt — 49.6 MB


In [11]:
import shutil, os

os.makedirs("/kaggle/working/download", exist_ok=True)

files_to_copy = [
    "best.pt",  # from weights folder
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxF1_curve.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
]

base = "/kaggle/working/runs/detect/helmet_kaggle_v1-2"

for f in files_to_copy:
    src = f"{base}/weights/{f}" if f == "best.pt" else f"{base}/{f}"
    shutil.copy(src, f"/kaggle/working/download/{f}")
    print(f"Copied: {f}")

# Zip it
shutil.make_archive("/kaggle/working/helmet_final", "zip", "/kaggle/working/download")
print("\nDone! Download helmet_final.zip from Output panel!")

Copied: best.pt
Copied: results.csv
Copied: results.png
Copied: confusion_matrix.png
Copied: confusion_matrix_normalized.png
Copied: BoxF1_curve.png
Copied: BoxPR_curve.png
Copied: BoxP_curve.png
Copied: BoxR_curve.png

Done! Download helmet_final.zip from Output panel!


In [10]:
import os
for f in os.listdir("/kaggle/working/runs/detect/helmet_kaggle_v1-2"):
    print(f)

results.csv
val_batch0_pred.jpg
train_batch2.jpg
BoxF1_curve.png
confusion_matrix.png
train_batch13922.jpg
train_batch13920.jpg
val_batch2_pred.jpg
confusion_matrix_normalized.png
BoxPR_curve.png
train_batch0.jpg
results.png
val_batch0_labels.jpg
weights
labels.jpg
BoxR_curve.png
train_batch13921.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
args.yaml
train_batch1.jpg
BoxP_curve.png


In [12]:
import os
for f in os.listdir("/kaggle/working"):
    print(f)

download
helmet_kaggle_results.zip
helmet-detection-2
runs
helmet_final.zip
yolov8m.pt
yolo26n.pt
.virtual_documents
